# Phase 2: Train Tiny TopK SAEs on Qwen3.5-35B-A3B Thinking Traces

**Input**: 41285 sentence-level L17 activations from `caiovicentino1/qwen35-a3b-thinking-traces` (Phase 1).

**Goal**: train TopK SAEs with n ∈ {5, 10, 15, 20, 25} per Venhoff et al. 2025 recipe. Pick elbow dict size. Save best to HF for Phase 3 (cluster labeling).

**SAE config** (exact recipe from paper, Appendix A):
- TopK activation, k=3
- Decoder normalized after each step
- Loss: plain MSE (sparsity via TopK, no L1 penalty)
- TinySAE lr schedule: `2e-4 / sqrt(n/16384)`
- Adam, batch 512, 300 epochs, early stop patience 10

**Budget**: ~10 min total (5 SAEs × ~2 min each — tiny!).

## Cell 1 — Setup + load activations

In [ ]:
import sys, subprocess, os
from pathlib import Path

def pip(*a): return subprocess.run([sys.executable, '-m', 'pip', *a], check=False)
try:
    import safetensors, huggingface_hub
except Exception:
    pip('install', '-q', 'safetensors', 'huggingface_hub==1.5.0', 'hf_transfer')

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        from huggingface_hub import login
        login(token=hf_token)
        print('HF auth OK')
except Exception as e:
    print(f'(skipping: {e})')

import json, time, pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from huggingface_hub import snapshot_download, HfApi
from safetensors.numpy import load_file
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

OUT = Path('/content/sae_phase2'); OUT.mkdir(exist_ok=True)
HF_PHASE1 = 'caiovicentino1/qwen35-a3b-thinking-traces'
HF_PHASE2 = 'caiovicentino1/qwen35-a3b-sae-phase2'

# Load Phase 1 data
path = snapshot_download(HF_PHASE1, repo_type='dataset', cache_dir='/content/cache')
data = load_file(str(Path(path) / 'activations.safetensors'))
activations = torch.from_numpy(data['activations'].astype(np.float32)).cuda()
prompt_idxs = data['prompt_idxs']
sentences = json.load(open(Path(path) / 'sentences.json'))
N, D = activations.shape
print(f'Loaded: {N} sentences × {D}-dim activations')
print(f'Unique prompts: {len(set(prompt_idxs))}')

## Cell 2 — TopK SAE (exact paper recipe)

In [ ]:
class TopKSAE(nn.Module):
    """TinySAE: TopK activation, MSE loss, decoder row-normed after each step."""
    def __init__(self, d_in, n_latents, k=3):
        super().__init__()
        self.d_in = d_in
        self.n = n_latents
        self.k = k
        self.W_enc = nn.Parameter(torch.randn(d_in, n_latents) * (1.0/d_in**0.5))
        self.b_enc = nn.Parameter(torch.zeros(n_latents))
        self.W_dec = nn.Parameter(torch.randn(n_latents, d_in) * (1.0/n_latents**0.5))
        self.b_dec = nn.Parameter(torch.zeros(d_in))
        with torch.no_grad():
            self.W_dec.data /= self.W_dec.data.norm(dim=-1, keepdim=True)
    def encode(self, x):
        pre = (x - self.b_dec) @ self.W_enc + self.b_enc
        topk_vals, topk_idx = pre.topk(self.k, dim=-1)
        z = torch.zeros_like(pre)
        z.scatter_(-1, topk_idx, F.relu(topk_vals))
        return z, topk_idx
    def decode(self, z):
        return z @ self.W_dec + self.b_dec
    def forward(self, x):
        z, idx = self.encode(x)
        return self.decode(z), z, idx

def train_sae(X, n_latents, k=3, epochs=300, batch_size=512, patience=10, seed=42):
    torch.manual_seed(seed)
    sae = TopKSAE(X.shape[1], n_latents, k).cuda()
    # TinySAE lr schedule
    lr = 2e-4 / (n_latents / 16384) ** 0.5
    opt = torch.optim.Adam(sae.parameters(), lr=lr, betas=(0.9, 0.999))
    with torch.no_grad():
        sae.b_dec.data = X.mean(dim=0)
    # Train/val split
    n_train = int(0.9 * X.shape[0])
    perm = torch.randperm(X.shape[0])
    X_train, X_val = X[perm[:n_train]], X[perm[n_train:]]
    best_val = float('inf'); wait = 0; history = []
    for epoch in range(epochs):
        sae.train()
        perm_ = torch.randperm(X_train.shape[0])
        losses = []
        for i in range(0, X_train.shape[0], batch_size):
            b = X_train[perm_[i:i+batch_size]]
            x_hat, z, _ = sae(b)
            loss = F.mse_loss(x_hat, b)
            opt.zero_grad(); loss.backward()
            # Project gradient off decoder rows to preserve unit norm
            with torch.no_grad():
                g = sae.W_dec.grad
                proj = (g * sae.W_dec.data).sum(dim=-1, keepdim=True) * sae.W_dec.data
                sae.W_dec.grad = g - proj
            opt.step()
            with torch.no_grad():
                sae.W_dec.data /= sae.W_dec.data.norm(dim=-1, keepdim=True).clamp(min=1e-8)
            losses.append(loss.item())
        sae.eval()
        with torch.no_grad():
            x_hat, _, _ = sae(X_val)
            val = F.mse_loss(x_hat, X_val).item()
        history.append(dict(epoch=epoch, train=np.mean(losses), val=val))
        if val < best_val - 1e-6:
            best_val = val; wait = 0
        else:
            wait += 1
            if wait >= patience: break
    return sae, history, best_val

print('✅ TopKSAE class ready')

## Cell 3 — Sweep dict sizes {5, 10, 15, 20, 25}

In [ ]:
DICT_SIZES = [5, 10, 15, 20, 25]
results = {}
t0 = time.time()

for n in DICT_SIZES:
    print(f'\n=== Training n={n} ===')
    ts = time.time()
    sae, history, best_val = train_sae(activations, n_latents=n, k=3, epochs=300)
    elapsed = time.time() - ts
    # Metrics
    with torch.no_grad():
        x_hat, z, idx = sae(activations)
        mse_all = F.mse_loss(x_hat, activations).item()
        var_explained = 1.0 - ((x_hat - activations).var(dim=0) / activations.var(dim=0).clamp(min=1e-8)).mean().item()
        cos_sim = F.cosine_similarity(x_hat, activations, dim=-1).mean().item()
        # Cluster size distribution (via top-1 index)
        top1 = idx[:, 0].cpu().numpy()
        cluster_sizes = np.bincount(top1, minlength=n)
    results[n] = dict(
        sae_state=sae.state_dict(),
        history=history,
        best_val=best_val,
        mse_all=mse_all,
        var_explained=var_explained,
        cos_sim=cos_sim,
        cluster_sizes=cluster_sizes.tolist(),
        elapsed=elapsed,
    )
    print(f'  n={n}: val MSE {best_val:.4f} | var exp {var_explained:.3f} | cos {cos_sim:.3f} | time {elapsed:.1f}s')
    print(f'  cluster sizes (top-1): {cluster_sizes.tolist()}')
    # Save SAE weights
    torch.save(sae.state_dict(), OUT / f'sae_n{n}.pt')

print(f'\n✅ Total sweep: {(time.time()-t0)/60:.1f} min')

## Cell 4 — Pick elbow + save best SAE

In [ ]:
import matplotlib.pyplot as plt

# Elbow detection: plot MSE vs n, pick n where second derivative peaks
ns = sorted(results.keys())
mses = [results[n]['best_val'] for n in ns]
vars_ = [results[n]['var_explained'] for n in ns]

print(f'{"n":>3} | {"val MSE":>8} | {"var_exp":>8} | {"cos":>5} | cluster sizes (top-1)')
for n in ns:
    r = results[n]
    cs = r['cluster_sizes']
    print(f'{n:>3d} | {r["best_val"]:>8.4f} | {r["var_explained"]:>7.3f} | {r["cos_sim"]:>5.3f} | {cs}')

# Elbow heuristic: smallest n where var_explained > 0.7, or where derivative drops
# Per paper: they picked n where grid-search elbow, varied by model (5, 10, 15, 20, 25).
# For simplicity, if var_exp jumps strongly between n and n+1, pick n+1
# Otherwise pick n=15 as default (matches 2 of paper's 5 models)
deltas = np.diff(vars_)
# Biggest jump = elbow just before
best_elbow_idx = np.argmax(deltas)
elbow_n = ns[best_elbow_idx + 1]
# Tie-break: if all deltas small, default n=15
if max(deltas) < 0.05:
    elbow_n = 15 if 15 in ns else ns[len(ns)//2]
    print(f'\nAll deltas small (<{0.05}), defaulting to n={elbow_n}')
else:
    print(f'\nBiggest var_exp jump: n={ns[best_elbow_idx]}→n={elbow_n} (+{deltas[best_elbow_idx]:.3f})')

print(f'\n🎯 Selected: n={elbow_n}')
print(f'  val MSE: {results[elbow_n]["best_val"]:.4f}')
print(f'  var explained: {results[elbow_n]["var_explained"]:.3f}')
print(f'  cos sim: {results[elbow_n]["cos_sim"]:.3f}')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(ns, mses, 'o-', label='val MSE')
axes[0].axvline(elbow_n, color='red', linestyle='--', label=f'elbow n={elbow_n}')
axes[0].set_xlabel('dict size n'); axes[0].set_ylabel('val MSE'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(ns, vars_, 'o-', color='green', label='var explained')
axes[1].axvline(elbow_n, color='red', linestyle='--')
axes[1].set_xlabel('dict size n'); axes[1].set_ylabel('variance explained'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT / 'sweep.png', dpi=120, bbox_inches='tight')
plt.show()

## Cell 5 — Extract cluster assignments + top-activating sentences per cluster

For the chosen n, save: per-cluster top-100 activating sentences + 100 random sentences. These go to Phase 3 (LLM labeling).

In [ ]:
BEST_N = elbow_n
sae = TopKSAE(D, BEST_N, k=3).cuda()
sae.load_state_dict(torch.load(OUT / f'sae_n{BEST_N}.pt'))
sae.eval()

with torch.no_grad():
    _, z_all, idx_all = sae(activations)  # z: [N, n], idx: [N, k]
    top1_latent = idx_all[:, 0].cpu().numpy()  # highest-activating latent per sentence
    top1_value = z_all.gather(-1, idx_all[:, 0:1]).squeeze(-1).cpu().numpy()

# For each cluster, get top-100 by activation + 100 random
import random
random.seed(42)
cluster_data = {}
for c in range(BEST_N):
    mask = (top1_latent == c)
    cluster_sentences = np.where(mask)[0]
    if len(cluster_sentences) == 0:
        cluster_data[c] = dict(top100=[], random100=[]); continue
    # Top by activation
    cluster_values = top1_value[cluster_sentences]
    top_sorted = cluster_sentences[np.argsort(-cluster_values)][:100]
    # Random
    rand_sample = random.sample(list(cluster_sentences), min(100, len(cluster_sentences)))
    cluster_data[c] = dict(
        top100=[sentences[i] for i in top_sorted],
        random100=[sentences[i] for i in rand_sample],
        size=int(mask.sum()))

for c in range(BEST_N):
    cd = cluster_data[c]
    print(f'Cluster {c}: {cd["size"]} sentences')
    if cd['top100']:
        print(f'  sample top: {cd["top100"][0][:120]}')

# Save
with open(OUT / f'cluster_data_n{BEST_N}.json', 'w') as f:
    json.dump(cluster_data, f, indent=2)
print(f'\n✅ cluster data saved for n={BEST_N}')

## Cell 6 — Upload SAE + cluster data to HF

In [ ]:
from huggingface_hub import create_repo

api = HfApi()
try:
    create_repo(HF_PHASE2, repo_type='model', private=False, exist_ok=True)
except: pass

# Save summary metrics (strip state_dicts to keep JSON readable)
summary = {n: {k: v for k, v in r.items() if k != 'sae_state'}
           for n, r in results.items()}
with open(OUT / 'summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)

# Upload all SAE checkpoints, best's cluster data, summary, plot
for n in DICT_SIZES:
    api.upload_file(path_or_fileobj=str(OUT / f'sae_n{n}.pt'),
                    path_in_repo=f'sae_n{n}.pt',
                    repo_id=HF_PHASE2, repo_type='model',
                    commit_message=f'SAE n={n}')
api.upload_file(path_or_fileobj=str(OUT / f'cluster_data_n{BEST_N}.json'),
                path_in_repo=f'cluster_data_n{BEST_N}.json',
                repo_id=HF_PHASE2, repo_type='model')
api.upload_file(path_or_fileobj=str(OUT / 'summary.json'),
                path_in_repo='summary.json',
                repo_id=HF_PHASE2, repo_type='model')
api.upload_file(path_or_fileobj=str(OUT / 'sweep.png'),
                path_in_repo='sweep.png',
                repo_id=HF_PHASE2, repo_type='model')

readme = f'''---
license: apache-2.0
base_model: Qwen/Qwen3.5-35B-A3B
tags:
  - mechanistic-interpretability
  - sparse-autoencoders
  - topk-sae
  - qwen3.5
---

# Qwen3.5-35B-A3B TopK SAEs — Phase 2 (Replication of Venhoff et al. 2025)

TopK Sparse Autoencoders trained on L17 residual-stream activations of `Qwen/Qwen3.5-35B-A3B` thinking traces.

## Recipe (Venhoff et al. 2025, arXiv:2510.07364)

- TopK activation, k=3
- Decoder row-normalized after each step
- MSE loss (sparsity via TopK, no L1 penalty)
- TinySAE lr schedule: `2e-4 / sqrt(n/16384)`
- Adam, batch 512, max 300 epochs, patience 10, 90/10 train/val split
- Activation source: sentence-level mean-pool (41285 sentences from 2000 MMLU-Pro prompts)

## Dict sizes swept

| n | val MSE | var explained | cos sim |
|---|---|---|---|
{chr(10).join(f"| {n} | {summary[n]['best_val']:.4f} | {summary[n]['var_explained']:.3f} | {summary[n]['cos_sim']:.3f} |" for n in DICT_SIZES)}

**Selected (elbow)**: n={BEST_N}

## Files

- `sae_n{{5,10,15,20,25}}.pt` — all trained SAEs
- `cluster_data_n{BEST_N}.json` — per-cluster top-100 + random-100 sentences (for LLM labeling in Phase 3)
- `summary.json` — training metrics per dict size
- `sweep.png` — MSE vs var-explained curves

## Next (Phase 3)

Label clusters with GPT-4o-mini using top-100 activating sentences → 10-20 named reasoning categories. Then train per-category steering vectors (Phase 4) and run hybrid inference on MATH500/GSM8K (Phase 5).

## Paper tracking

Full replication pipeline: https://huggingface.co/datasets/caiovicentino1/Qwen3.6-35B-A3B-mcr-stage-b

License: Apache-2.0.
'''
with open(OUT / 'README.md', 'w') as f: f.write(readme)
api.upload_file(path_or_fileobj=str(OUT / 'README.md'),
                path_in_repo='README.md',
                repo_id=HF_PHASE2, repo_type='model')
print(f'✅ uploaded to https://huggingface.co/{HF_PHASE2}')